# Test Option B — deepagents + serveur MCP via `uvx` depuis GitHub

**Option B** : aucun `git clone` manuel. `uvx` récupère et exécute le paquet
`projeqtor-mcp-server-python` directement depuis le dépôt GitHub, en transport
**stdio**. Les tools MCP sont chargés dans un agent **deepagents** (LangChain)
piloté par un modèle **AWS Bedrock**.

```
deepagents (LangChain) ── tools ──> MultiServerMCPClient
        │ Bedrock                          │ stdio
        ▼                          uvx --from git+github ... projeqtor-mcp-server-python
   ChatBedrockConverse                      │ httpx
                                      API REST ProjeQtOr
```

## Prérequis

- `uv` installé et `uvx` sur le PATH (https://docs.astral.sh/uv/)
- Accès AWS Bedrock (modèle activé) + credentials AWS
- Instance ProjeQtOr joignable + credentials
- Dépôt privé : `uvx` utilise ton auth git locale (SSH ou token)

## 1. Dépendances

In [1]:
# %pip install deepagents langchain-mcp-adapters langchain-aws python-dotenv

## 2. Configuration

Reprend le `.env` (PROJEQTOR_* + AWS). Ces variables sont passées au
sous-processus serveur lancé par `uvx`.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# --- ProjeQtOr ---
os.environ.setdefault("PROJEQTOR_BASE_URL", "https://mon-instance.projeqtor.com")
os.environ.setdefault("PROJEQTOR_API_KEY", "ma_cle_api_aes")
os.environ.setdefault("PROJEQTOR_AES_KEY_LENGTH", "128")
os.environ.setdefault("LOG_LEVEL", "WARNING")
# Auth ProjeQtOr : PROJEQTOR_BEARER_TOKEN, ou PROJEQTOR_USERNAME + PROJEQTOR_PASSWORD

# --- AWS Bedrock ---
AWS_REGION = os.environ.setdefault("AWS_REGION", "us-east-1")
BEDROCK_MODEL_ID = os.environ.setdefault("BEDROCK_MODEL_ID", "anthropic.claude-3-5-sonnet-20241022-v2:0")

REPO = "git+https://github.com/CouLiBaLy-B/server-mcp-ipmp.git"

print("Région AWS    :", AWS_REGION)
print("Modèle Bedrock:", BEDROCK_MODEL_ID)
print("ProjeQtOr URL :", os.environ["PROJEQTOR_BASE_URL"])

Région AWS    : us-east-1
Modèle Bedrock: zai.glm-5
ProjeQtOr URL : https://ipmp-beta.siinergy.net


## 3. Workaround Windows + Jupyter (`fileno`)

`MultiServerMCPClient` lance le serveur via `stdio_client`, qui passe
`sys.stderr` comme `errlog` au sous-processus. Sous ipykernel, `sys.stderr`
n'expose pas `fileno()` → `UnsupportedOperation: fileno`. `StdioConnection`
n'expose pas `errlog`, donc on bascule temporairement `sys.stderr` vers un vrai
fichier autour des appels qui spawnent le serveur.

Sous Linux/macOS, le helper est inoffensif (no-op pratique).

In [3]:
import contextlib
import sys


@contextlib.contextmanager
def real_stderr(path: str = "mcp_server.log"):
    """Bascule sys.stderr vers un vrai fichier (avec fileno) le temps du bloc."""
    f = open(path, "a", encoding="utf-8")
    orig = sys.stderr
    sys.stderr = f
    try:
        yield
    finally:
        sys.stderr = orig
        f.close()

## 4. Client MCP — serveur lancé par `uvx` (Option B)

`env` inclut tout `os.environ` (notamment `PATH`, sinon `uvx` introuvable) plus
les overrides ProjeQtOr/transport.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "projeqtor": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["--from", REPO, "projeqtor-mcp-server-python"],
            "env": {**os.environ, "MCP_TRANSPORT": "stdio"},
        }
    }
)

# Premier appel : uvx télécharge le paquet depuis GitHub (peut prendre 10-60 s).
with real_stderr():
    tools = await client.get_tools()

print(f"{len(tools)} tools chargés via uvx :")
for t in tools[:10]:
    print("  -", t.name)

ModuleNotFoundError: No module named 'langchain_mcp_adapters'

## 5. Modèle Bedrock + agent deepagents

In [ ]:
from langchain_aws import ChatBedrockConverse
from deepagents import create_deep_agent

model = ChatBedrockConverse(model=BEDROCK_MODEL_ID, region_name=AWS_REGION, temperature=0)

agent = create_deep_agent(
    tools=tools,
    model=model,
    instructions=(
        "Assistant de gestion de projet ProjeQtOr connecté via des tools MCP. "
        "Utilise les tools pour répondre. Réponds en français, de façon concise."
    ),
)
print("Agent prêt.")

## 6. Exécution

Chaque appel de tool relance le serveur via `uvx` (stdio éphémère), d'où le
`real_stderr()` autour de `ainvoke`. Nécessite une instance ProjeQtOr réelle
pour des données.

In [ ]:
QUESTION = "Liste les projets disponibles et résume le premier."

with real_stderr():
    result = await agent.ainvoke({"messages": [{"role": "user", "content": QUESTION}]})

print(result["messages"][-1].content)

## Dépannage

- **`uvx: command not found`** : installer `uv` (https://docs.astral.sh/uv/) et vérifier qu'il est sur le PATH du kernel. `env` doit contenir `PATH` (déjà via `{**os.environ}`).
- **`UnsupportedOperation: fileno`** : entourer l'appel de `with real_stderr():` (§3).
- **`uvx` clone échoue (dépôt privé)** : configurer l'auth git locale (clé SSH ou token dans l'URL/credential helper).
- **`AccessDeniedException` Bedrock** : activer le modèle (*Model access*) ou utiliser un inference profile (`us.anthropic...`).
- **Premier appel lent** : `uvx` build l'environnement au 1er run, puis cache. Pour figer une révision : `--from git+...@<tag-ou-sha>`.
- **Aucune donnée** : vérifier `PROJEQTOR_BASE_URL` + auth + clé AES.